# Classroom Emotion Recognition - Best Colab Pipeline

This notebook is the recommended Colab training flow for the highest accuracy in this project.

Pipeline:
1. Install dependencies and enable GPU-friendly settings.
2. Upload `kaggle.json` and download DAiSEE and optional FER2013.
3. Prepare 4-class labels.
4. Extract DAiSEE frames with face-first crop.
5. Pretrain EfficientNetV2B0 on FER2013.
6. Fine-tune on DAiSEE.
7. Export the best model and result images.

Recommended runtime: `GPU` in Google Colab.


In [ ]:
import os
import random
import subprocess
from pathlib import Path

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('\n'.join(result.stdout.splitlines()[:5]))
else:
    print('GPU not found. In Colab go to Runtime > Change runtime type > GPU')

!pip install -q kaggle opencv-python-headless scikit-learn seaborn tqdm

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print('Environment ready.')


In [ ]:
from google.colab import files

print('Upload kaggle.json from your Kaggle account settings.')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

USE_FER_PRETRAIN = True
FRAMES_PER_CLIP = 10
IMG_SIZE = 260
BATCH_SIZE = 32
FER_EPOCHS_HEAD = 6
FER_EPOCHS_FINETUNE = 6
DAISEE_EPOCHS_HEAD = 10
DAISEE_EPOCHS_FINETUNE = 14

BASE_DIR = Path('/content')
RAW_DAISEE = BASE_DIR / 'daisee_raw'
RAW_FER = BASE_DIR / 'fer2013_raw'
FER_PREP = BASE_DIR / 'fer2013_4class'
DAISEE_PREP = BASE_DIR / 'daisee_4class'
RUN_DIR = BASE_DIR / 'best_emotion_run'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration ready.')


In [ ]:
print('Downloading DAiSEE...')
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip

if USE_FER_PRETRAIN:
    print('Downloading FER2013...')
    !kaggle datasets download -d msambare/fer2013 -p /content/fer2013_raw --unzip

!find /content/daisee_raw -maxdepth 2 -type d | head -20
if USE_FER_PRETRAIN:
    !find /content/fer2013_raw -maxdepth 2 -type d | head -20


In [ ]:
import glob
import json
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

EMOTION_LABELS_VI = ['Buon chan', 'Tap trung', 'Hung thu', 'Binh thuong']
FER2013_TO_EMOTION = {
    'angry': 0,
    'disgust': 0,
    'fear': 0,
    'sad': 0,
    'happy': 2,
    'surprise': 2,
    'neutral': 3,
}
AUTOTUNE = tf.data.AUTOTUNE

tf.random.set_seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision enabled.')
    except Exception as exc:
        print('Could not enable mixed precision:', exc)

CASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
FACE_CASCADE = cv2.CascadeClassifier(CASCADE_PATH)

def map_daisee_to_label(row):
    boredom = row.get('Boredom', row.get('boredom', 0))
    engagement = row.get('Engagement', row.get('engagement', 0))
    confusion = row.get('Confusion', row.get('confusion', 0))
    frustration = row.get('Frustration', row.get('frustration', 0))
    if boredom >= 2 and engagement <= 1:
        return 0
    if engagement >= 2 and confusion >= 1 and frustration < 2:
        return 2
    if engagement >= 2:
        return 1
    return 3

def ensure_class_dirs(root_dir, splits):
    for split in splits:
        for class_idx in range(4):
            (root_dir / split / str(class_idx)).mkdir(parents=True, exist_ok=True)

def reset_split_dirs(root_dir, splits):
    if root_dir.exists():
        shutil.rmtree(root_dir)
    ensure_class_dirs(root_dir, splits)

def compute_class_weights_from_dir(split_dir):
    labels = []
    for class_idx in range(4):
        image_count = len(list((split_dir / str(class_idx)).glob('*.jpg')))
        labels.extend([class_idx] * image_count)
    labels = np.array(labels, dtype=np.int32)
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
    if 0 in class_weights:
        class_weights[0] *= 1.4
    if 2 in class_weights:
        class_weights[2] *= 1.15
    if 3 in class_weights:
        class_weights[3] *= 1.4
    for idx in range(4):
        class_weights[idx] = max(class_weights.get(idx, 1.0), 0.35)
    return class_weights

def build_loss():
    focal = getattr(tf.keras.losses, 'CategoricalFocalCrossentropy', None)
    if focal is not None:
        return focal(alpha=0.3, gamma=2.0)
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

def build_model(img_size=260, num_classes=4):
    inputs = tf.keras.Input(shape=(img_size, img_size, 3), name='image')
    x = tf.keras.layers.RandomFlip('horizontal')(inputs)
    x = tf.keras.layers.RandomRotation(0.08)(x)
    x = tf.keras.layers.RandomZoom(0.12)(x)
    x = tf.keras.layers.RandomTranslation(0.08, 0.08)(x)
    x = tf.keras.layers.RandomContrast(0.15)(x)
    x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
    backbone = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
        pooling='avg',
    )
    backbone.trainable = False
    x = tf.keras.layers.BatchNormalization()(backbone.output)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(256, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32', name='emotion')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='emotion_efficientnetv2b0')
    model.backbone = backbone
    return model

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=build_loss(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
        ],
    )

def set_backbone_trainable(model, trainable_fraction):
    backbone = getattr(model, 'backbone', None)
    if backbone is None:
        backbone = next((layer for layer in model.layers if isinstance(layer, tf.keras.Model) and 'efficientnet' in layer.name), None)
    if backbone is None:
        raise ValueError('Could not locate EfficientNet backbone on the model.')
    backbone.trainable = True
    freeze_until = int(len(backbone.layers) * (1.0 - trainable_fraction))
    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False
    for layer in backbone.layers[freeze_until:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True

def build_callbacks(prefix):
    return [
        tf.keras.callbacks.ModelCheckpoint(str(CHECKPOINT_DIR / f'{prefix}.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / f'{prefix}.csv')),
    ]

def make_dataset(root_dir, split, img_size=260, batch_size=32, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        root_dir / split,
        label_mode='categorical',
        image_size=(img_size, img_size),
        batch_size=batch_size,
        shuffle=shuffle,
        seed=SEED,
    ).prefetch(AUTOTUNE)

def find_csv(csv_files, keyword):
    matches = [path for path in csv_files if keyword.lower() in path.lower()]
    return matches[0] if matches else None

def find_video_path(clip_name):
    candidates = [
        f'/content/daisee_raw/**/{clip_name}',
        f'/content/daisee_raw/**/{clip_name}.avi',
        f'/content/daisee_raw/**/{clip_name}.mp4',
    ]
    for pattern in candidates:
        found = glob.glob(pattern, recursive=True)
        if found:
            return found[0]
    return None

def crop_face_or_center(frame, pad=24):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))
    if len(faces) > 0:
        areas = [w * h for (_, _, w, h) in faces]
        x, y, w, h = faces[int(np.argmax(areas))]
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(frame.shape[1], x + w + pad)
        y2 = min(frame.shape[0], y + h + pad)
        return frame[y1:y2, x1:x2]
    h, w = frame.shape[:2]
    side = min(h, w)
    x1 = (w - side) // 2
    y1 = (h - side) // 2
    return frame[y1:y1 + side, x1:x1 + side]

def extract_frames(video_path, label, split, clip_id, frames_per_clip=10, img_size=260):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return 0
    sample_points = np.linspace(0, total_frames - 1, frames_per_clip, dtype=int)
    saved = 0
    for frame_idx, sample_idx in enumerate(sample_points):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(sample_idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        crop = crop_face_or_center(frame)
        crop = cv2.resize(crop, (img_size, img_size))
        out_path = DAISEE_PREP / split / str(label) / f'{clip_id}_{frame_idx}.jpg'
        cv2.imwrite(str(out_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 94])
        saved += 1
    cap.release()
    return saved

def prepare_fer2013():
    reset_split_dirs(FER_PREP, ['train', 'val'])
    split_aliases = {'train': {'train', 'training'}, 'val': {'test', 'validation', 'val'}}
    for split_name, aliases in split_aliases.items():
        split_dir = None
        for child in RAW_FER.rglob('*'):
            if child.is_dir() and child.name.lower() in aliases:
                split_dir = child
                break
        if split_dir is None:
            raise FileNotFoundError(f'FER2013 split not found for {split_name}')
        for emotion_dir in split_dir.iterdir():
            if not emotion_dir.is_dir():
                continue
            mapped = FER2013_TO_EMOTION.get(emotion_dir.name.lower())
            if mapped is None:
                continue
            destination = FER_PREP / split_name / str(mapped)
            for image_path in emotion_dir.rglob('*'):
                if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                    continue
                target = destination / image_path.name
                if target.exists():
                    target = destination / f'{image_path.stem}_{abs(hash(str(image_path))) % 100000}{image_path.suffix.lower()}'
                shutil.copy2(image_path, target)

def prepare_daisee(frames_per_clip=10, img_size=260):
    reset_split_dirs(DAISEE_PREP, ['train', 'val', 'test'])
    csv_files = glob.glob('/content/daisee_raw/**/*.csv', recursive=True)
    df_train = pd.read_csv(find_csv(csv_files, 'train'))
    df_val = pd.read_csv(find_csv(csv_files, 'validat'))
    df_test = pd.read_csv(find_csv(csv_files, 'test'))
    stats = {'train': 0, 'val': 0, 'test': 0}
    for df, split in [(df_train, 'train'), (df_val, 'val'), (df_test, 'test')]:
        df['emotion_label'] = df.apply(map_daisee_to_label, axis=1)
        print(split, df['emotion_label'].value_counts().to_dict())
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'Extract {split}'):
            clip_name = row.get('ClipID', row.get('Clip', row.get('clip', None)))
            if clip_name is None:
                continue
            clip_name = str(clip_name)
            video_path = find_video_path(clip_name)
            if video_path is None:
                continue
            stats[split] += extract_frames(video_path, int(row['emotion_label']), split, idx, frames_per_clip=frames_per_clip, img_size=img_size)
    print('Extracted frames:', stats)

def count_images(root_dir):
    summary = {}
    for split in ['train', 'val', 'test']:
        if not (root_dir / split).exists():
            continue
        split_counts = {}
        for class_idx in range(4):
            split_counts[class_idx] = len(list((root_dir / split / str(class_idx)).glob('*.jpg')))
        summary[split] = split_counts
    return summary

def plot_training_curves(histories, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for stage_name, history in histories.items():
        if 'accuracy' in history.history:
            axes[0].plot(history.history['accuracy'], label=f'{stage_name} train')
        if 'val_accuracy' in history.history:
            axes[0].plot(history.history['val_accuracy'], linestyle='--', label=f'{stage_name} val')
        if 'loss' in history.history:
            axes[1].plot(history.history['loss'], label=f'{stage_name} train')
        if 'val_loss' in history.history:
            axes[1].plot(history.history['val_loss'], linestyle='--', label=f'{stage_name} val')
    axes[0].set_title('Accuracy')
    axes[1].set_title('Loss')
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.set_xlabel('Epoch')
        axis.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()

def evaluate_and_save(model, test_ds):
    y_true_all = []
    y_pred_all = []
    for images, labels in test_ds:
        probs = model.predict(images, verbose=0)
        y_true_all.append(np.argmax(labels.numpy(), axis=1))
        y_pred_all.append(np.argmax(probs, axis=1))
    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    report = classification_report(y_true, y_pred, labels=[0, 1, 2, 3], target_names=EMOTION_LABELS_VI, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_LABELS_VI, yticklabels=EMOTION_LABELS_VI)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(RUN_DIR / 'confusion_matrix.png', dpi=160)
    plt.show()
    with open(RUN_DIR / 'classification_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    return report

print('Utility functions ready.')


In [ ]:
if USE_FER_PRETRAIN:
    prepare_fer2013()
    print('FER2013 prepared.')
    print(count_images(FER_PREP))

prepare_daisee(frames_per_clip=FRAMES_PER_CLIP, img_size=IMG_SIZE)
print('DAiSEE prepared.')
print(count_images(DAISEE_PREP))


In [ ]:
histories = {}
model = build_model(img_size=IMG_SIZE)

if USE_FER_PRETRAIN:
    fer_train = make_dataset(FER_PREP, 'train', img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True)
    fer_val = make_dataset(FER_PREP, 'val', img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
    fer_class_weights = compute_class_weights_from_dir(FER_PREP / 'train')
    print('FER class weights:', fer_class_weights)

    compile_model(model, learning_rate=8e-4)
    histories['fer_head'] = model.fit(
        fer_train,
        validation_data=fer_val,
        epochs=FER_EPOCHS_HEAD,
        class_weight=fer_class_weights,
        callbacks=build_callbacks('fer_head'),
        verbose=1,
    )

    set_backbone_trainable(model, trainable_fraction=0.25)
    compile_model(model, learning_rate=1e-4)
    histories['fer_finetune'] = model.fit(
        fer_train,
        validation_data=fer_val,
        epochs=FER_EPOCHS_FINETUNE,
        class_weight=fer_class_weights,
        callbacks=build_callbacks('fer_finetune'),
        verbose=1,
    )

daisee_train = make_dataset(DAISEE_PREP, 'train', img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True)
daisee_val = make_dataset(DAISEE_PREP, 'val', img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
daisee_test = make_dataset(DAISEE_PREP, 'test', img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
daisee_class_weights = compute_class_weights_from_dir(DAISEE_PREP / 'train')
print('DAiSEE class weights:', daisee_class_weights)

compile_model(model, learning_rate=7e-4)
histories['daisee_head'] = model.fit(
    daisee_train,
    validation_data=daisee_val,
    epochs=DAISEE_EPOCHS_HEAD,
    class_weight=daisee_class_weights,
    callbacks=build_callbacks('daisee_head'),
    verbose=1,
)

set_backbone_trainable(model, trainable_fraction=0.4)
compile_model(model, learning_rate=7e-5)
histories['daisee_finetune'] = model.fit(
    daisee_train,
    validation_data=daisee_val,
    epochs=DAISEE_EPOCHS_FINETUNE,
    class_weight=daisee_class_weights,
    callbacks=build_callbacks('daisee_finetune'),
    verbose=1,
)

print('Training finished.')


In [ ]:
best_checkpoint = CHECKPOINT_DIR / 'daisee_finetune.keras'
if best_checkpoint.exists():
    best_model = tf.keras.models.load_model(best_checkpoint)
else:
    best_model = model

report = evaluate_and_save(best_model, daisee_test)
plot_training_curves(histories, RUN_DIR / 'training_curves.png')

final_model_path = RUN_DIR / 'emotion_model_daisee_best.h5'
best_model.save(final_model_path)

summary = {
    'config': {
        'use_fer_pretrain': USE_FER_PRETRAIN,
        'frames_per_clip': FRAMES_PER_CLIP,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
    },
    'dataset_summary': count_images(DAISEE_PREP),
    'report': report,
}
with open(RUN_DIR / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Saved model:', final_model_path)
print('Saved report:', RUN_DIR / 'training_summary.json')
print('Saved confusion matrix:', RUN_DIR / 'confusion_matrix.png')
print('Saved training curves:', RUN_DIR / 'training_curves.png')


In [ ]:
%cd /content
!zip -r best_emotion_run.zip best_emotion_run >/dev/null
from google.colab import files
files.download('/content/best_emotion_run.zip')
